In [3]:
import os
import json
import random
from copy import deepcopy
from importlib import reload
from glob import glob

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

import experiment as E
from models.simple_classifier import model as M, utils as U, config as C, analysis as A, train as T, loss as L, evaluation as Ev, infer as I
from data_readers import data_loader as D, radar as R, infrared as IR, name_map as N
import genetic_search as G
_ = reload(M), reload(U), reload(C), reload(A), reload(D), reload(T), reload(R), reload(IR), reload(N), reload(E), reload(G), reload(L), reload(Ev), reload(I)

torch.set_printoptions(precision=4, sci_mode=False)
# x, y = A.get_dummy_xy()
# {k:x.shape for k,x in x.items()}

DATASET_PATH = os.path.abspath("../../../RF-Behavior")
EXPERIMENTS_DIR = os.path.abspath("./experiments")
# LOGS_FILENAME = "experiment_logs.jsonl"

RADAR_BIN_FPS = 18.7
BEHAVIOR_COMBOS = [sorted(combo) for combo in [
    # ("M01", "M02"),
    # ("M12", "M13", "M20", "M21"),
    # ("M04", "M05", "M12", "M13"),
    N.MOTION_MAP.keys(),
    N.ACTIVITY_MAP.keys(),
    N.EMOTION_MAP.keys(),
    list(N.MOTION_MAP.keys()) + list(N.ACTIVITY_MAP.keys()),
    list(N.MOTION_MAP.keys()) + list(N.ACTIVITY_MAP.keys()) + list(N.EMOTION_MAP.keys()),
]]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
LABELS = BEHAVIOR_COMBOS[-1]
print(LABELS)
LOGS_FILENAME = f"{len(LABELS)}_class_experiment_logs.jsonl"
# LOGS_FILENAME = f"experiment_logs.jsonl"
LOGS_FILENAME

['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'M01', 'M02', 'M03', 'M04', 'M05', 'M06', 'M07', 'M08', 'M09', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M19', 'M20', 'M21']


'37_class_experiment_logs.jsonl'

In [ ]:
for _ in range(1):
    E.run_experiments(
        labels=LABELS, lr=1e-3, patience=5, max_epochs=100, batch_size=32, use_genetic=False,
        device=DEVICE, dataset_dir=DATASET_PATH, experiments_dir=EXPERIMENTS_DIR,
        logs_filename=LOGS_FILENAME, model_variants_path="./model_variants.csv",
)

model_df.shape=(4906, 5) | buckets per modality: {'infrared': 20, 'radar': 20}
modality_to_logs: {'radar': 20, 'infrared': 20}
data_df.shape=(7261, 6) NA: {'infrared_path': 3, 'radar_path': 2}
len(common_users)=12:U03,U07,U11,U14,U21,U22,U32,U33,U35,U38,U41,U44


Preloading dataset:   0%|          | 0/5484 [00:00<?, ?sample/s]

Preloading dataset:   0%|          | 0/592 [00:00<?, ?sample/s]

Preloading dataset:   0%|          | 0/1183 [00:00<?, ?sample/s]

Loaded data for modality='radar' | len(train_dataset)=5484, len(valid_dataset)=592, len(test_dataset)=1183 | valid_dataset.users=['U11', 'U41'], test_dataset.users=['U07', 'U14', 'U32', 'U35'] | ','.join(train_dataset.label_names)='A01,A02,A03,A04,A05,A06,A07,A08,A09,A10,E01,E02,E03,E04,E05,E06,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11,M12,M13,M14,M15,M16,M17,M18,M19,M20,M21', ','.join(valid_dataset.label_names)='A01,A02,A03,A04,A05,A06,A07,A08,A09,A10,E01,E02,E03,E04,E05,E06,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11,M12,M13,M14,M15,M16,M17,M18,M19,M20,M21', ','.join(test_dataset.label_names)='A01,A02,A03,A04,A05,A06,A07,A08,A09,A10,E01,E02,E03,E04,E05,E06,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11,M12,M13,M14,M15,M16,M17,M18,M19,M20,M21'
LR set in log: 0.001
Batch size set in log: 32
[1] Running experiment for RadarEncoder with 82485 parameters and config={
  "input_dim": 4,
  "output_dim": 37,
  "point_cloud_encoder_kwargs": {
    "input_dim": 4,
    "embedding_size": 16,
    "n

## Unsupervised Model Training

### Contrastive - Single Modality

In [4]:
BEHAVIOR_SPLITS = [
    (sorted(set(trn) - set(val+tst)), sorted(set(val)), sorted(set(tst)))
    for trn, val, tst in [

    (sorted(N.MOTION_MAP  .keys()), ["M02", "M03"], ["M04", "M05"]),
    (sorted(N.ACTIVITY_MAP.keys()), ["A03", "A04"], ["A05", "A06"]),
    (sorted(N.MOTION_MAP  .keys())+sorted(N.ACTIVITY_MAP.keys()), ["M02", "M03", "A09", "A10"], ["M04", "M05", "A05", "A06"]),
    (sorted(N.MOTION_MAP  .keys())+sorted(N.ACTIVITY_MAP.keys())+sorted(N.EMOTION_MAP.keys()), ["M02", "M03", "A09", "A10", "E05", "E06"], ["M04", "M05", "A05", "A06", "E03", "E04"]),
]]


TRAIN_BEHAVIORS, VALID_BEHAVIORS, TEST_BEHAVIORS = BEHAVIOR_SPLITS[0]
print(f"{TRAIN_BEHAVIORS=},\n{VALID_BEHAVIORS=},\n {TEST_BEHAVIORS=}")

TRAIN_BEHAVIORS=['M01', 'M06', 'M07', 'M08', 'M09', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M19', 'M20', 'M21'],
VALID_BEHAVIORS=['M02', 'M03'],
 TEST_BEHAVIORS=['M04', 'M05']


In [5]:
LOGS_FILENAME = f"{len(TRAIN_BEHAVIORS)}_cat_contrastive_exp_logs.jsonl"
LOGS_FILENAME

'17_cat_contrastive_exp_logs.jsonl'

In [ ]:
for _ in range(1):
    E.run_experiments(
        objective="contrastive", labels=(TRAIN_BEHAVIORS, VALID_BEHAVIORS, TEST_BEHAVIORS),
        lr=1e-3, patience=5, max_epochs=100, batch_size=1, use_genetic=False,
        device=DEVICE, dataset_dir=DATASET_PATH, experiments_dir=EXPERIMENTS_DIR,
        logs_filename=LOGS_FILENAME, model_variants_path="./model_variants.csv",

        # skip_radar=True,
)

model_df.shape=(4906, 5) | buckets per modality: {'infrared': 20, 'radar': 20}
modality_to_logs: {'radar': 20, 'infrared': 20}
data_df.shape=(7261, 6) NA: {'infrared_path': 3, 'radar_path': 2}
len(common_users)=12:U03,U07,U11,U14,U21,U22,U32,U33,U35,U38,U41,U44


Preloading dataset:   0%|          | 0/4926 [00:00<?, ?sample/s]

Preloading dataset:   0%|          | 0/1146 [00:00<?, ?sample/s]

Preloading dataset:   0%|          | 0/1186 [00:00<?, ?sample/s]

Loaded data for modality='infrared' | len(train_dataset)=4926, len(valid_dataset)=1146, len(test_dataset)=1186 | valid_dataset.users=['U01', 'U03', 'U04', 'U05', 'U06', 'U07', 'U08', 'U10', 'U11', 'U12', 'U13', 'U14', 'U17', 'U19', 'U21', 'U22', 'U24', 'U25', 'U28', 'U29', 'U30', 'U32', 'U33', 'U35', 'U36', 'U38', 'U41', 'U42', 'U44'], test_dataset.users=['U01', 'U03', 'U04', 'U05', 'U06', 'U07', 'U08', 'U10', 'U11', 'U12', 'U13', 'U14', 'U17', 'U19', 'U21', 'U22', 'U24', 'U25', 'U28', 'U29', 'U30', 'U32', 'U33', 'U35', 'U36', 'U38', 'U41', 'U42', 'U44'] | ','.join(train_dataset.label_names)='A01,A02,A03,A04,A07,A08,E01,E02,M01,M06,M07,M08,M09,M10,M11,M12,M13,M14,M15,M16,M17,M18,M19,M20,M21', ','.join(valid_dataset.label_names)='A09,A10,E05,E06,M02,M03', ','.join(test_dataset.label_names)='A05,A06,E03,E04,M04,M05'
LR set in log: 0.001
Batch size set in log: 1
[1] Running experiment for InfraredEncoder with 85184 parameters and config={
  "input_dim": 42,
  "output_dim": 256,
  "downsam

In [ ]:
valid= ["push-down", "lift", "kick-floor-ball", "kick-football", "depression", "excitement"]
test = ["pull", "push", "ascending-stairs", "descending-stairs", "stress", "relaxation"]

### Cross-Modal

In [6]:
LOGS_FILENAME = f"{len(TRAIN_BEHAVIORS)}_cat_cross_modal_exp_logs.jsonl"
LOGS_FILENAME

'17_cat_cross_modal_exp_logs.jsonl'

## Extra

In [3]:
df = D.find_available_files("../../../RF-Behavior")
df["name"] = df.behavior.apply(N.map_to_name)
df.groupby(["campaign", "behavior", "name"]).size().reset_index(name="count").sort_values(["campaign", "count", "behavior"])#.groupby("count")

,campaign,behavior,name,count
14,C1,M15,left-arm-circle,186
15,C1,M16,right-arm-circle,186
18,C1,M19,two-hand-lateral-to-front,186
19,C1,M20,circle-clockwise,186
20,C1,M21,circle-counter-clockwise,186
7,C1,M08,swipe-left,193
8,C1,M09,throw,193
9,C1,M10,arms-swing,202
10,C1,M11,two-hand-throw,202
11,C1,M12,two-hand-push,202
